#### Configuration
One TinyStories continuation and a coherence judgment, or the saved demonstration.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / 'llm' / 'model.py').is_file())
sys.path.insert(0, str(REPO_ROOT))

RUN = False
MODEL_NAME = 'roneneldan/TinyStories-28M'
DTYPE, DEVICE_MAP = 'auto', 'cuda:0'
PROMPT = 'Once upon a time'
MAX_NEW_TOKENS, SEED = 128, 0
ARTIFACTS = REPO_ROOT / 'artifacts' / MODEL_NAME.rsplit('/', 1)[-1].lower() / '00_test_load'
RESULTS_PATH = ARTIFACTS / 'sample.csv'

#### Model
Load the language model only when generating a new demonstration.

In [2]:
if RUN:
    import torch
    from llm.model import load_model
    from llm.steering import sample
    from llm.judge import build_judge, COHERENCE

    torch.manual_seed(SEED)
    tokenizer, model = load_model(MODEL_NAME, dtype=DTYPE, device_map=DEVICE_MAP)

#### Demonstration
Save the continuation and its binary coherence judgment, or read the saved CSV.

In [3]:
if RUN:
    ids = sample(model, tokenizer, PROMPT, n=1, max_new_tokens=MAX_NEW_TOKENS)
    prompt_length = len(tokenizer(PROMPT)['input_ids'])
    text = tokenizer.decode(ids[0, prompt_length:], skip_special_tokens=True)
    tokens = ids.shape[1] - prompt_length
    del model
    torch.cuda.empty_cache()
    judge = build_judge(device_map=DEVICE_MAP)
    coherent = judge(COHERENCE, PROMPT, [text])[0]
    results = pd.DataFrame([dict(prompt=PROMPT, completion=text,
                                 generated_tokens=tokens, coherent=coherent)])
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    results.to_csv(RESULTS_PATH, index=False)
    del judge
else:
    results = pd.read_csv(RESULTS_PATH)

#### Summary
The saved text, token count, and judgment describe this single sample only.

In [4]:
with pd.option_context('display.max_colwidth', None):
    display(results)

,prompt,completion,generated_tokens,coherent
0,Once upon a time,"there was a little girl called Sally. She loved to play in the park with her friends, but one day she found something very unusual - an old rag! It looked like it had been washed by someone else and it felt so soft when touched it.\n\nSally picked up the rag and ran home as fast as she could. When she got back to her house, she put the rag on the kitchen table. Then she grabbed some crayons from the drawer and started to draw lots of different shapes and patterns.\n\nThe next morning, Sally showed the rag to her mum who said that it must have belonged to somebody",128,True


#### Result
TinyStories produced 128 new tokens, and Granite judged the continuation coherent. This single sample checks loading, generation and judging, not population-level model quality.